預先側兩筆

In [1]:
from google import genai
import os
import time
import json
import pandas as pd

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

SAFETY_LABELS = [
    "safe",
    "illness",
    "contamination",
    "spoilage",
    "allergen",
    "quality_defect",
]

LABEL_DEFINITIONS = {
    "safe": "The review does not describe any food safety hazard, illness, contamination, spoilage, allergen issue, or product defect.",
    "illness": "The consumer or pet became sick after eating the product, such as vomiting, diarrhea, food poisoning, nausea, stomach pain, or infection.",
    "contamination": "The product contained foreign objects or harmful substances, such as mold, insects, worms, maggots, glass, plastic, metal, chemicals, dirt, or other physical contamination.",
    "spoilage": "The product was spoiled, expired, rotten, rancid, moldy, fermented, putrid, smelled bad, bulging, leaking, or otherwise unsafe due to freshness problems.",
    "allergen": "The product caused an allergic reaction or contained undisclosed allergens, such as nuts, dairy, gluten, soy, shellfish, or other hidden ingredients.",
    "quality_defect": "The product had a defect or safety-related quality issue, such as broken packaging, broken glass, damaged seal, undercooked food, tampering, wrong label, misleading packaging, or hard object causing injury risk."
}

reviews = [
    {
        "Id": 1,
        "Text": "I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than most."
    },
    {
        "Id": 2,
        "Text": "Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as \"Jumbo\"."
    }
]

def build_prompt(review_id, text):
    return f"""
You are a food safety review classifier.

Classify the review into exactly ONE category based on the following label definitions.

Label definitions:

Safe:
"is safe and helped the consumer recover from previous health issues"
Use this when the review describes no food safety problem, or describes the product as safe, healthy, beneficial, or helpful.

Illness:
"caused severe illness, food poisoning, or sickness"
Use this when the review says a person or pet became sick after consuming the product.
Examples: food poisoning, vomiting, diarrhea, nausea, stomach pain, salmonella, infection, sickness.

Contamination:
"contained physical contamination, mold, insects, or foreign objects"
Use this when the review mentions contamination inside the product.
Examples: insects, worms, maggots, glass shards, plastic pieces, metal pieces, mold, fungus, dirt, chemicals, bleach, soap.

Spoilage:
"was spoiled, expired, rotten, or smelled bad"
Use this when the review says the product was spoiled, expired, rotten, rancid, putrid, smelled bad, bulging, bloated, leaking, or unsafe due to freshness.

Allergen:
"triggered a severe allergic reaction or contained undisclosed allergens"
Use this when the review mentions allergic reaction or hidden allergens.
Examples: anaphylactic shock, throat swelling, breathing difficulty, hives, skin reaction, undisclosed allergens.

Quality_Defect:
"had packaging defects, broken glass, or undercooked food"
Use this when the review describes packaging defects or product defects.
Examples: broken seal, tampered seal, damaged packaging, broken glass, undercooked food, hard dangerous object, chipped tooth.

Reference probes:
- Contracted severe food poisoning and vomiting → Illness
- Diagnosed with salmonella gastroenteritis infection → Illness
- Intense nausea and violent diarrhea all night → Illness
- Disgusting maggots and worms crawling inside → Contamination
- Sharp glass shards and hard plastic pieces hidden → Contamination
- Tasted like toxic chemical bleach and soap → Contamination
- Smells completely rancid, rotten, and putrid → Spoilage
- Covered in fuzzy green mold and fungus → Contamination
- The can was bulging, bloated, and leaking → Spoilage
- Anaphylactic shock, throat swelling, and breathing difficulty → Allergen
- Severe allergic reaction and skin hives breakout → Allergen
- Dangerous undisclosed hidden allergens ingredients → Allergen
- Chipped my tooth on a hard dangerous object → Quality_Defect
- Tampered broken safety seal risk → Quality_Defect

Priority rule:
If multiple categories appear, choose the most serious one:
Illness > Allergen > Contamination > Spoilage > Quality_Defect > Safe

Important rules:
- Return exactly one category.
- The category must be one of:
  Safe, Illness, Contamination, Spoilage, Allergen, Quality_Defect
- Return valid JSON only.
- Do not output markdown.
- Do not explain.

Output format:
{{
  "Id": {review_id},
  "label": "one_category_only"
}}

Review:
Id: {review_id}
Text: {text}
"""

inline_requests = []
for row in reviews:
    inline_requests.append({
        "contents": [
            {
                "role": "user",
                "parts": [{"text": build_prompt(row["Id"], row["Text"])}]
            }
        ],
        "config": {
            "temperature": 0,
            "response_mime_type": "application/json"
        }
    })

batch_job = client.batches.create(
    model="gemini-3-flash-preview",
    src=inline_requests,
    config={
        "display_name": "food-safety-batch"
    }
)

print("Batch job created:", batch_job.name)
print("Initial state:", batch_job.state)

job_name = batch_job.name

while True:
    job = client.batches.get(name=job_name)
    state = job.state.name if hasattr(job.state, "name") else str(job.state)
    print("Current state:", state)

    if state in [
        "JOB_STATE_SUCCEEDED",
        "JOB_STATE_FAILED",
        "JOB_STATE_CANCELLED",
        "JOB_STATE_EXPIRED"
    ]:
        break

    time.sleep(5)

results = []

if state == "JOB_STATE_SUCCEEDED":
    for res in job.dest.inlined_responses:
        try:
            text = res.response.text.strip()
            data = json.loads(text)

            results.append({
                "Id": data["Id"],
                "label": data["label"]
            })

        except Exception as e:
            results.append({
                "Id": None,
                "label": "ERROR"
            })

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("Id").reset_index(drop=True)

    print(result_df)

    result_df.to_csv(
        "food_safety_labels.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("CSV 已輸出：food_safety_labels.csv")

else:
    print("Batch job did not succeed.")

Batch job created: batches/8jusdy5u4fio07f7endvtnw8farda0fohflz
Initial state: JobState.JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Cur

##########幫我從這裡 執行

In [ ]:
!pip install google-generativeai -q

In [ ]:
!pip install pandas -q

In [2]:
import pandas as pd

# 讀取資料（假設是 CSV）
df = pd.read_csv("Reviews_withURL.csv")

# 隨機抽樣 5000 筆
sample_df = df.sample(n=5000, random_state=42)

# ⭐ 清理 Text（避免 NaN）
sample_df["Text"] = sample_df["Text"].fillna("").astype(str)

# 只保留指定欄位
sample_df = sample_df[["Id", "ProductId", "Text"]]

print("抽樣完成:", sample_df.shape)
print(sample_df.head())

抽樣完成: (5000, 3)
            Id   ProductId                                               Text
165256  165257  B000EVG8J2  Having tried a couple of other brands of glute...
231465  231466  B0000BXJIS  My cat loves these treats. If ever I can't fin...
427827  427828  B008FHUFAU  A little less than I expected.  It tends to ha...
433954  433955  B006BXV14E  First there was Frosted Mini-Wheats, in origin...
70260    70261  B007I7Z3Z0  and I want to congratulate the graphic artist ...


In [3]:
sample_reviews = sample_df[["Id", "Text"]].to_dict("records")
display(sample_reviews[:5])

[{'Id': 165257,
  'Text': 'Having tried a couple of other brands of gluten-free sandwich cookies, these are the best of the bunch.  They\'re crunchy and true to the texture of the other "real" cookies that aren\'t gluten-free.  Some might think that the filling makes them a bit too sweet, but for me that just means I\'ve satisfied my sweet tooth sooner!  The chocolate version from Glutino is just as good and has a true "chocolatey" taste - something that isn\'t there with the other gluten-free brands out there.'},
 {'Id': 231466,
  'Text': "My cat loves these treats. If ever I can't find her in the house, I just pop the top and she bolts out of wherever she was hiding to come get a treat. She doesn't like crunchy treats much, so these are perfect for her. I've given her all three flavors and she seems to like them all equally. They do tend to dry out by the time I near the end of the bottle, however. The flip-top lid is very handy. Very nice, inexpensive kitty treats. I have yet to mee

正式執行

In [ ]:
import pandas as pd
from google import genai
import os
import time
import json



# =========================
# 2. Gemini client
# =========================
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# 直接用 sample_reviews
reviews = sample_reviews

def build_prompt(review_id, text):
    return f"""
You are a food safety review classifier.

Classify the review into exactly ONE category based on the following label definitions.

Label definitions:

Safe:
"is safe and helped the consumer recover from previous health issues"
Use this when the review describes no food safety problem, or describes the product as safe, healthy, beneficial, or helpful.

Illness:
"caused severe illness, food poisoning, or sickness"
Use this when the review says a person or pet became sick after consuming the product.
Examples: food poisoning, vomiting, diarrhea, nausea, stomach pain, salmonella, infection, sickness.

Contamination:
"contained physical contamination, mold, insects, or foreign objects"
Use this when the review mentions contamination inside the product.
Examples: insects, worms, maggots, glass shards, plastic pieces, metal pieces, mold, fungus, dirt, chemicals, bleach, soap.

Spoilage:
"was spoiled, expired, rotten, or smelled bad"
Use this when the review says the product was spoiled, expired, rotten, rancid, putrid, smelled bad, bulging, bloated, leaking, or unsafe due to freshness.

Allergen:
"triggered a severe allergic reaction or contained undisclosed allergens"
Use this when the review mentions allergic reaction or hidden allergens.
Examples: anaphylactic shock, throat swelling, breathing difficulty, hives, skin reaction, undisclosed allergens.

Quality_Defect:
"had packaging defects, broken glass, or undercooked food"
Use this when the review describes packaging defects or product defects.
Examples: broken seal, tampered seal, damaged packaging, broken glass, undercooked food, hard dangerous object, chipped tooth.

Reference probes:
- Contracted severe food poisoning and vomiting → Illness
- Diagnosed with salmonella gastroenteritis infection → Illness
- Intense nausea and violent diarrhea all night → Illness
- Disgusting maggots and worms crawling inside → Contamination
- Sharp glass shards and hard plastic pieces hidden → Contamination
- Tasted like toxic chemical bleach and soap → Contamination
- Smells completely rancid, rotten, and putrid → Spoilage
- Covered in fuzzy green mold and fungus → Contamination
- The can was bulging, bloated, and leaking → Spoilage
- Anaphylactic shock, throat swelling, and breathing difficulty → Allergen
- Severe allergic reaction and skin hives breakout → Allergen
- Dangerous undisclosed hidden allergens ingredients → Allergen
- Chipped my tooth on a hard dangerous object → Quality_Defect
- Tampered broken safety seal risk → Quality_Defect

Priority rule:
If multiple categories appear, choose the most serious one:
Illness > Allergen > Contamination > Spoilage > Quality_Defect > Safe

Important rules:
- Return exactly one category.
- The category must be one of:
  Safe, Illness, Contamination, Spoilage, Allergen, Quality_Defect
- Return valid JSON only.
- Do not output markdown.
- Do not explain.

Output format:
{{
  "Id": {review_id},
  "label": "one_category_only"
}}

Review:
Id: {review_id}
Text: {text}
"""

# =========================
# 3. 建立 inline requests
# =========================
inline_requests = []
for row in reviews:
    inline_requests.append({
        "contents": [
            {
                "role": "user",
                "parts": [{"text": build_prompt(row["Id"], row["Text"])}]
            }
        ],
        "config": {
            "temperature": 0,
            "response_mime_type": "application/json"
        }
    })

# =========================
# 4. 建立 Batch job
# =========================
batch_job = client.batches.create(
    model="gemini-3-flash-preview",
    src=inline_requests,
    config={
        "display_name": "food-safety-batch"
    }
)

print("Batch job created:", batch_job.name)
print("Initial state:", batch_job.state)

# =========================
# 5. 輪詢直到完成
# =========================
job_name = batch_job.name

while True:
    job = client.batches.get(name=job_name)
    state = job.state.name if hasattr(job.state, "name") else str(job.state)
    print("Current state:", state)

    if state in [
        "JOB_STATE_SUCCEEDED",
        "JOB_STATE_FAILED",
        "JOB_STATE_CANCELLED",
        "JOB_STATE_EXPIRED"
    ]:
        break

    time.sleep(5)

# =========================
# 6. 解析結果
# =========================
results = []

if state == "JOB_STATE_SUCCEEDED":
    for res in job.dest.inlined_responses:
        try:
            text = res.response.text.strip()
            data = json.loads(text)

            results.append({
                "Id": data["Id"],
                "label": data["label"]
            })
        except Exception:
            results.append({
                "Id": None,
                "label": "ERROR"
            })

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("Id").reset_index(drop=True)

    print(result_df.head())

    result_df.to_csv("food_safety_labels.csv", index=False, encoding="utf-8-sig")
    print("CSV 已輸出：food_safety_labels.csv")

else:
    print("Batch job did not succeed.")

In [ ]:
import pandas as pd

# 讀取 label 結果
label_df = pd.read_csv("food_safety_labels.csv")

# 合併回原資料
final_df = sample_df.merge(label_df, on="Id", how="left")
# 新增欄位 human_is_hazard
final_df["human_is_hazard"] = final_df["label"].apply(lambda x: 0 if x == "safe" else 1)
final_df.to_csv("sample_with_labels.csv", index=False, encoding="utf-8-sig")

print(final_df.head())